# 伤病预测模型：未来是否会受伤

## 目标
基于球员的历史伤病史、位置、身高、出场数据等特征，预测**指定球员**在**指定天数**内是否会受伤（二分类）。

## 本节内容
1. **数据加载**：读取 `./saved/players_filled.csv`，解析 `injury_history`
2. **标签构造**：从伤病史时间轴生成 (观察点, 预测窗口, 是否受伤) 样本

## 1. 数据加载

In [1]:
import pandas as pd
import numpy as np
import ast

# 加载 players_filled.csv（PART3 已预处理，height 已填充缺失值）
df_raw = pd.read_csv('./saved/players_filled.csv')

def parse_injury_history(s):
    """将 injury_history 字符串解析为整数列表。偶数位=健康天数，奇数位=伤病天数。"""
    if pd.isna(s) or s == '':
        return []
    if isinstance(s, list):
        return [int(x) if isinstance(x, (int, float)) else int(str(x).strip().replace(' days', '')) for x in s]
    s = str(s).strip()
    try:
        arr = ast.literal_eval(s)
        return [int(x) if isinstance(x, (int, float)) else int(str(x).strip().replace(' days', '')) for x in arr]
    except (ValueError, SyntaxError):
        return []

df_raw['injury_series'] = df_raw['injury_history'].map(parse_injury_history)

print(f"加载了 {len(df_raw)} 个球员的数据")
print(f"列: {list(df_raw.columns)}")
print(f"\n示例 injury_series（Ederson 前 12 个）: {df_raw.iloc[0]['injury_series'][:12]}")
df_raw.head()

加载了 2591 个球员的数据
列: ['name', 'position', 'height/m', 'nationality', 'injury_history', 'apparence', 'injury_series']

示例 injury_series（Ederson 前 12 个）: [-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 17]


,name,position,height/m,nationality,injury_history,apparence,injury_series
0,Ederson,Goalkeeper,1.88,Brazil,"[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...","(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
1,Stefan Ortega,Goalkeeper,1.85,Germany,"[-1, 54, 447, 4, 101, 5, 2411, 28, 451, 8, 666...","(401, 35735)","[-1, 54, 447, 4, 101, 5, 2411, 28, 451, 8, 666..."
2,Scott Carson,Goalkeeper,1.88,England,"[-1, 9, 34, 17, 2837, 68]","(522, 46879)","[-1, 9, 34, 17, 2837, 68]"
3,Rúben Dias,Centre-Back,1.87,Portugal,"[-1, 29, 95, 11, 1435, 40, 255, 25, 654, 33, 1...","(425, 36453)","[-1, 29, 95, 11, 1435, 40, 255, 25, 654, 33, 1..."
4,Abdukodir Khusanov,Centre-Back,1.86,Uzbekistan,[-1],"(78, 6292)",[-1]


## 2. 标签构造

标签样本：<br>
在每条球员的伤病史时间轴上，从每个**健康期起点**作为观察点，给定预测窗口 `horizon_days`（如 30、60、90 天）：
- 若从该起点起的 `horizon_days` 天内**会进入伤病段** → 标签 **1**（受伤）
- 若该段时间内**始终处于健康段** → 标签 **0**（不受伤）

首段健康期为 `-1`（未知长度）时跳过该观察点。

In [2]:
def build_label_samples_for_player(name, series, horizon_days_list=(30, 60, 90)):
    """
    从一名球员的 injury_series 生成标签样本。
    series: 交替 [健康天数, 伤病天数, 健康天数, ...]，偶数下标为健康段。
    仅在「健康段起点」生成样本；该段长度为 healthy_days 时，
    若 healthy_days < horizon_days 则未来 horizon_days 天内会受伤 → 1，否则 → 0。
    """
    if not series or len(series) < 2:   # 实际上伤病长度为 2的也不会被记录，因为其并没有非-1的偶数位
        return []
    samples = []
    for i in range(0, len(series), 2):  # 偶数位：健康段
        healthy_days = series[i]
        if healthy_days == -1:  # 首段未知，跳过
            continue
        for horizon_days in horizon_days_list:
            label = 1 if healthy_days < horizon_days else 0
            samples.append({
                'name': name,
                'horizon_days': horizon_days,
                'segment_idx': i,
                'label': label,
            })
    return samples

# 为所有球员生成标签样本（使用 30 / 60 / 90 天三种预测窗口）
HORIZON_DAYS_LIST = list(range(10, 101, 10))
rows = []
for _, player in df_raw.iterrows():
    for s in build_label_samples_for_player(player['name'], player['injury_series'], HORIZON_DAYS_LIST):
        rows.append(s)

df_labels = pd.DataFrame(rows)
print(f"共生成 {len(df_labels)} 条标签样本（球员数×健康段数×预测窗口数）")
print(f"列: {list(df_labels.columns)}")
print(f"\n标签分布:\n{df_labels['label'].value_counts()}")
print(f"\n按 horizon_days 的标签分布:")
print(df_labels.groupby('horizon_days')['label'].value_counts().unstack(fill_value=0))
df_labels.head(15)

共生成 164650 条标签样本（球员数×健康段数×预测窗口数）
列: ['name', 'horizon_days', 'segment_idx', 'label']

标签分布:
label
0    113141
1     51509
Name: count, dtype: int64

按 horizon_days 的标签分布:
label             0     1
horizon_days             
10            14729  1736
20            13653  2812
30            12830  3635
40            12017  4448
50            11330  5135
60            10721  5744
70            10230  6235
80             9707  6758
90             9198  7267
100            8726  7739


,name,horizon_days,segment_idx,label
0,Ederson,10,2,0
1,Ederson,20,2,0
2,Ederson,30,2,0
3,Ederson,40,2,0
4,Ederson,50,2,0
5,Ederson,60,2,0
6,Ederson,70,2,0
7,Ederson,80,2,0
8,Ederson,90,2,0
9,Ederson,100,2,0


In [3]:
# 将标签与原始球员信息合并，得到「样本表」（后续特征工程基于此表）
df_samples = df_labels.merge(
    df_raw[['name', 'position', 'height/m', 'nationality', 'apparence', 'injury_series']],
    on='name', how='left'
)
print(f"样本表形状: {df_samples.shape}")
print(f"列: {list(df_samples.columns)}")
df_samples.head(10)

样本表形状: (164970, 9)
列: ['name', 'horizon_days', 'segment_idx', 'label', 'position', 'height/m', 'nationality', 'apparence', 'injury_series']


,name,horizon_days,segment_idx,label,position,height/m,nationality,apparence,injury_series
0,Ederson,10,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
1,Ederson,20,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
2,Ederson,30,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
3,Ederson,40,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
4,Ederson,50,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
5,Ederson,60,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
6,Ederson,70,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
7,Ederson,80,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
8,Ederson,90,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."
9,Ederson,100,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1..."


___
### 标签构造测试

In [4]:
player1 = df_raw.iloc[0]
player2 = df_raw.iloc[-2]
print('ederson:', player1['injury_series'])
print('ederson:', player2['injury_series'])

ederson: [-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 17, 1096, 8, 50, 27, 39, 76, 145, 22, 22, 10, 32, 17, 14, 13]
ederson: [-1, 124, 34, 91]


In [5]:
player1

name                                                        Ederson
position                                                 Goalkeeper
height/m                                                       1.88
nationality                                                  Brazil
injury_history    [-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...
apparence                                              (495, 44035)
injury_series     [-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...
Name: 0, dtype: object

In [6]:
player2

name                  Sidiki Chérif
position             Centre-Forward
height/m                       1.85
nationality                  France
injury_history    [-1, 124, 34, 91]
apparence                (53, 3194)
injury_series     [-1, 124, 34, 91]
Name: 2589, dtype: object

In [7]:
build_label_samples_for_player(player2['name'], player2['injury_series'], (30,))

[{'name': 'Sidiki Chérif', 'horizon_days': 30, 'segment_idx': 2, 'label': 0}]

Ederson 伤病波纹图：
<figure>
    <left> <img src="images/edersonwave.png"  alt='missing' width="500"  ><left/>
<figure/>

In [8]:
build_label_samples_for_player(player1['name'], player1['injury_series'], (30,))

[{'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 2, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 4, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 6, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 8, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 10, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 12, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 14, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 16, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 18, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 20, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 22, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 30, 'segment_idx': 24, 'label': 1}]

In [9]:
build_label_samples_for_player(player1['name'], player1['injury_series'], (10,))

[{'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 2, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 4, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 6, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 8, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 10, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 12, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 14, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 16, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 18, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 20, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 22, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 10, 'segment_idx': 24, 'label': 0}]

In [10]:
build_label_samples_for_player(player1['name'], player1['injury_series'], (90,))

[{'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 2, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 4, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 6, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 8, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 10, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 12, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 14, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 16, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 18, 'label': 0},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 20, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 22, 'label': 1},
 {'name': 'Ederson', 'horizon_days': 90, 'segment_idx': 24, 'label': 1}]

In [11]:
df_labels

,name,horizon_days,segment_idx,label
0,Ederson,10,2,0
1,Ederson,20,2,0
2,Ederson,30,2,0
3,Ederson,40,2,0
4,Ederson,50,2,0
...,...,...,...,...
164645,Sidiki Chérif,60,2,1
164646,Sidiki Chérif,70,2,1
164647,Sidiki Chérif,80,2,1
164648,Sidiki Chérif,90,2,1


---

## 3. 特征工程

基于技术文档，为每条样本构造特征：
- **预测条件**：`horizon_days`
- **静态**：`position`、`height`、`nationality`（编码）
- **出场**：从 `apparence` 解析 `total_games`、`total_minutes`、`avg_minutes_per_game`
- **伤病史（到观察点）**：`segment_idx` 之前的历史 → `num_injuries`、`total_injury_days`、`avg_injury_days`、`max_injury_days`、`last_injury_days`、`has_injury_history`
- **近期**：近期若干健康段/伤病段的均值、标准差

In [12]:
def parse_apparence(s):
    """从 apparence 字符串 '(场次, 总分钟)' 解析为 (total_games, total_minutes)。"""
    if pd.isna(s):
        return 0, 0
    s = str(s).strip().strip('()')
    try:
        parts = s.split(',')
        games = int(parts[0].strip())
        minutes = int(parts[1].strip())
        return games, minutes
    except (ValueError, IndexError):
        return 0, 0

def injury_history_features_up_to(series, segment_idx):
    """
    根据 injury_series 与观察点 segment_idx，计算到该观察点为止的伤病史特征。
    segment_idx 为当前健康段的偶数下标；只使用 series[0 : segment_idx] 的历史。
    """
    if segment_idx <= 0:
        return {
            'num_injuries': 0,
            'total_injury_days': 0,
            'avg_injury_days': 0.0,
            'max_injury_days': 0,
            'last_injury_days': 0,
            'has_injury_history': 0,
            'recent_healthy_days_mean': np.nan,
            'recent_healthy_days_std': np.nan,
            'recent_injury_days_mean': np.nan,
            'recent_injury_days_std': np.nan,
        }
    # 观察点之前的伤病段：奇数下标 1, 3, ..., segment_idx-1
    injury_days_list = [series[i] for i in range(1, segment_idx, 2) if i < len(series)]
    num_injuries = len(injury_days_list)
    total_injury_days = sum(injury_days_list) if injury_days_list else 0
    avg_injury_days = total_injury_days / num_injuries if num_injuries else 0.0
    max_injury_days = max(injury_days_list) if injury_days_list else 0
    last_injury_days = series[segment_idx - 1] if segment_idx >= 1 and segment_idx - 1 < len(series) else 0

    # 观察点之前的健康段（偶数下标 0, 2, ..., segment_idx-2），排除 -1，取最近最多 5 段
    healthy_days_list = [series[i] for i in range(0, segment_idx, 2) if i < len(series) and series[i] != -1][-5:]
    recent_healthy_mean = np.mean(healthy_days_list) if healthy_days_list else np.nan
    recent_healthy_std = np.std(healthy_days_list) if len(healthy_days_list) > 1 else np.nan

    recent_injury_mean = np.mean(injury_days_list[-5:]) if injury_days_list else np.nan
    recent_injury_std = np.std(injury_days_list[-5:]) if len(injury_days_list) >= 2 else np.nan

    return {
        'num_injuries': num_injuries,
        'total_injury_days': total_injury_days,
        'avg_injury_days': avg_injury_days,
        'max_injury_days': max_injury_days,
        'last_injury_days': last_injury_days,
        'has_injury_history': 1 if num_injuries > 0 else 0,
        'recent_healthy_days_mean': recent_healthy_mean,
        'recent_healthy_days_std': recent_healthy_std,
        'recent_injury_days_mean': recent_injury_mean,
        'recent_injury_days_std': recent_injury_std,
    }

In [13]:
# 解析出场数据
apparence_parsed = df_samples['apparence'].map(parse_apparence)
df_samples['total_games'] = [x[0] for x in apparence_parsed]
df_samples['total_minutes'] = [x[1] for x in apparence_parsed]
df_samples['avg_minutes_per_game'] = df_samples.apply(
    lambda r: r['total_minutes'] / r['total_games'] if r['total_games'] > 0 else 0, axis=1
)
df_samples

,name,horizon_days,segment_idx,label,position,height/m,nationality,apparence,injury_series,total_games,total_minutes,avg_minutes_per_game
0,Ederson,10,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...",495,44035,88.959596
1,Ederson,20,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...",495,44035,88.959596
2,Ederson,30,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...",495,44035,88.959596
3,Ederson,40,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...",495,44035,88.959596
4,Ederson,50,2,0,Goalkeeper,1.88,Brazil,"(495, 44035)","[-1, 30, 397, 3, 785, 17, 16, 7, 16, 9, 355, 1...",495,44035,88.959596
...,...,...,...,...,...,...,...,...,...,...,...,...
164965,Sidiki Chérif,60,2,1,Centre-Forward,1.85,France,"(53, 3194)","[-1, 124, 34, 91]",53,3194,60.264151
164966,Sidiki Chérif,70,2,1,Centre-Forward,1.85,France,"(53, 3194)","[-1, 124, 34, 91]",53,3194,60.264151
164967,Sidiki Chérif,80,2,1,Centre-Forward,1.85,France,"(53, 3194)","[-1, 124, 34, 91]",53,3194,60.264151
164968,Sidiki Chérif,90,2,1,Centre-Forward,1.85,France,"(53, 3194)","[-1, 124, 34, 91]",53,3194,60.264151


In [14]:
df_samples['total_games'] = [x[0] for x in apparence_parsed]
df_samples['total_minutes'] = [x[1] for x in apparence_parsed]
df_samples['avg_minutes_per_game'] = df_samples.apply(
    lambda r: r['total_minutes'] / r['total_games'] if r['total_games'] > 0 else 0, axis=1
)

# 到观察点为止的伤病史特征
injury_feats = df_samples.apply(
    lambda r: injury_history_features_up_to(r['injury_series'], r['segment_idx']), axis=1
)
for key in injury_feats.iloc[0].keys():
    df_samples[key] = injury_feats.map(lambda d: d[key])

# 身高（沿用 height/m）
df_samples['height'] = df_samples['height/m'].astype(float)

In [15]:
# 类别编码：position 用 One-Hot，nationality 用 LabelEncoder（高基数）
from sklearn.preprocessing import LabelEncoder

position_dummies = pd.get_dummies(df_samples['position'], prefix='pos')
le_nat = LabelEncoder()
df_samples['nationality_enc'] = le_nat.fit_transform(df_samples['nationality'].astype(str).fillna('Unknown'))

# 近期统计的缺失值用 0 填充（表示无历史）
for col in ['recent_healthy_days_mean', 'recent_healthy_days_std', 'recent_injury_days_mean', 'recent_injury_days_std']:
    df_samples[col] = df_samples[col].fillna(0)

# 合并为特征表
df_feat = pd.concat([
    df_samples[['horizon_days', 'height', 'total_games', 'total_minutes', 'avg_minutes_per_game',
                 'num_injuries', 'total_injury_days', 'avg_injury_days', 'max_injury_days',
                 'last_injury_days', 'has_injury_history',
                 'recent_healthy_days_mean', 'recent_healthy_days_std',
                 'recent_injury_days_mean', 'recent_injury_days_std', 'nationality_enc']],
    position_dummies
], axis=1)

# 目标变量
y = df_samples['label'].values
feature_cols = list(df_feat.columns)
X = df_feat.values

print("特征维度:", X.shape)
print("特征名:", feature_cols)
print("标签分布:", pd.Series(y).value_counts().to_dict())
df_feat.head()

特征维度: (164970, 29)
特征名: ['horizon_days', 'height', 'total_games', 'total_minutes', 'avg_minutes_per_game', 'num_injuries', 'total_injury_days', 'avg_injury_days', 'max_injury_days', 'last_injury_days', 'has_injury_history', 'recent_healthy_days_mean', 'recent_healthy_days_std', 'recent_injury_days_mean', 'recent_injury_days_std', 'nationality_enc', 'pos_Attacking Midfield', 'pos_Central Midfield', 'pos_Centre-Back', 'pos_Centre-Forward', 'pos_Defensive Midfield', 'pos_Goalkeeper', 'pos_Left Midfield', 'pos_Left Winger', 'pos_Left-Back', 'pos_Right Midfield', 'pos_Right Winger', 'pos_Right-Back', 'pos_Second Striker']
标签分布: {0: 113347, 1: 51623}


,horizon_days,height,total_games,total_minutes,avg_minutes_per_game,num_injuries,total_injury_days,avg_injury_days,max_injury_days,last_injury_days,...,pos_Centre-Forward,pos_Defensive Midfield,pos_Goalkeeper,pos_Left Midfield,pos_Left Winger,pos_Left-Back,pos_Right Midfield,pos_Right Winger,pos_Right-Back,pos_Second Striker
0,10,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
1,20,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
2,30,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
3,40,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
4,50,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False


In [16]:
# 特征工程结果汇总：样本数、特征数、缺失与类型
print(f"样本数: {len(df_feat)}, 特征数: {len(feature_cols)}")
print(f"特征缺失: {df_feat.isna().sum().sum()}")
print("\n数值特征描述:")
df_feat.describe().round(2)

样本数: 164970, 特征数: 29
特征缺失: 0

数值特征描述:


,horizon_days,height,total_games,total_minutes,avg_minutes_per_game,num_injuries,total_injury_days,avg_injury_days,max_injury_days,last_injury_days,has_injury_history,recent_healthy_days_mean,recent_healthy_days_std,recent_injury_days_mean,recent_injury_days_std,nationality_enc
count,164970.00,164970.00,164970.00,164970.00,164970.00,164970.00,164970.00,164970.00,164970.00,164970.00,164970.0,164970.00,164970.00,164970.00,164970.00,164970.00
mean,55.00,1.83,348.54,25124.15,70.89,7.99,224.15,32.38,85.74,30.59,1.0,205.26,127.56,31.58,20.77,48.46
std,28.72,0.07,147.18,12054.71,10.62,7.34,207.41,29.67,77.12,43.95,0.0,240.96,149.71,30.86,25.51,27.71
min,10.00,1.63,27.00,1026.00,34.11,1.00,0.00,0.00,0.00,0.00,1.0,0.00,0.00,0.00,0.00,0.00
25%,30.00,1.78,243.00,16552.00,63.16,3.00,66.00,16.11,32.00,8.00,1.0,65.60,17.13,13.60,4.85,30.00
50%,55.00,1.84,335.00,23631.00,71.38,6.00,160.00,24.60,63.00,15.00,1.0,139.60,85.95,22.60,12.24,37.00
75%,80.00,1.88,433.00,31727.00,78.87,11.00,323.00,38.06,109.00,33.00,1.0,260.20,175.82,38.00,26.60,77.00
max,100.00,2.06,952.00,85656.00,91.21,70.00,1579.00,561.00,561.00,561.00,1.0,3652.00,1790.00,561.00,238.03,102.00


In [17]:
df_feat

,horizon_days,height,total_games,total_minutes,avg_minutes_per_game,num_injuries,total_injury_days,avg_injury_days,max_injury_days,last_injury_days,...,pos_Centre-Forward,pos_Defensive Midfield,pos_Goalkeeper,pos_Left Midfield,pos_Left Winger,pos_Left-Back,pos_Right Midfield,pos_Right Winger,pos_Right-Back,pos_Second Striker
0,10,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
1,20,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
2,30,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
3,40,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
4,50,1.88,495,44035,88.959596,1,30,30.0,30,30,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164965,60,1.85,53,3194,60.264151,1,124,124.0,124,124,...,True,False,False,False,False,False,False,False,False,False
164966,70,1.85,53,3194,60.264151,1,124,124.0,124,124,...,True,False,False,False,False,False,False,False,False,False
164967,80,1.85,53,3194,60.264151,1,124,124.0,124,124,...,True,False,False,False,False,False,False,False,False,False
164968,90,1.85,53,3194,60.264151,1,124,124.0,124,124,...,True,False,False,False,False,False,False,False,False,False


---
## 4. 划分数据集

按**球员**划分训练集 / 验证集 / 测试集，确保同一球员的所有样本只出现在一个集合中，避免信息泄漏。比例示例：70% / 15% / 15%。

In [18]:
from sklearn.model_selection import train_test_split

# 按球员名划分：先得到唯一球员列表，再按比例拆成 train / val / test
unique_players = df_samples['name'].unique()
n_players = len(unique_players)
np.random.seed(42)
np.random.shuffle(unique_players)

# 70% 训练 / 15% 验证 / 15% 测试
n_train = int(0.70 * n_players)
n_val = int(0.15 * n_players)
train_players = set(unique_players[:n_train])
val_players = set(unique_players[n_train : n_train + n_val])
test_players = set(unique_players[n_train + n_val :])

# 根据样本所属球员打上集合标签
train_mask = df_samples['name'].isin(train_players)
val_mask = df_samples['name'].isin(val_players)
test_mask = df_samples['name'].isin(test_players)

# X,y 的训练、验证、测试集
X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]

print(f"唯一球员数: {n_players} → 训练 {len(train_players)} / 验证 {len(val_players)} / 测试 {len(test_players)}")
print(f"样本数: 训练 {len(y_train)} / 验证 {len(y_val)} / 测试 {len(y_test)}")
print(f"训练集标签分布: {pd.Series(y_train).value_counts().to_dict()}")

唯一球员数: 1994 → 训练 1395 / 验证 299 / 测试 300
样本数: 训练 114050 / 验证 25300 / 测试 25620
训练集标签分布: {0: 78470, 1: 35580}


In [19]:
train_mask

0          True
1          True
2          True
3          True
4          True
          ...  
164965    False
164966    False
164967    False
164968    False
164969    False
Name: name, Length: 164970, dtype: bool

In [20]:
train_players

{'Leny Yoro',
 'Marc Cucurella',
 'Sam Morsy',
 'Adrian Beck',
 'Mateo Retegui',
 'Matías Soulé',
 'Florian Wirtz',
 'Théo Sainte-Luce',
 'Marius Bülter',
 'Yordan Osorio',
 'Erencan Yardımcı',
 'Leif Davis',
 'Amad Diallo',
 'Daniel Batz',
 'Jamal Musiala',
 'Grigoris Kastanos',
 'Benjamin Siegrist',
 'Alexandre Lacazette',
 'Ben Davies',
 'Alberto Moreno',
 'Calvin Bassey',
 'Domen Crnigoj',
 'Stanislav Lobotka',
 'Nicola Zalewski',
 'Anass Salah-Eddine',
 'Martin Valjent',
 'José Gayà',
 'Saša Lukić',
 'Oriol Romeu',
 'Nico Elvedi',
 'Samuele Ricci',
 'Marco Silvestri',
 'Marius Wolf',
 'Rafa Marín',
 'Yasin Ayari',
 'Arnau Tenas',
 'Aritz Elustondo',
 'Fabian Rieder',
 'Carles Aleñá',
 'Armin Gigovic',
 'Zakaria Aboukhlal',
 'Valentin Rongier',
 'Nathan Patterson',
 'Emmanuel Gyasi',
 'Tanguy Nianzou',
 'Julio Enciso',
 'Willian Pacho',
 'Tino Kadewere',
 'Martin Terrier',
 'Oscar Højlund',
 'Cheick Doucouré',
 'André Almeida',
 'Bjarki Bjarkason',
 'Warmed Omari',
 'Carlos Augusto

In [21]:
X_train

array([[10, 1.88, 495, ..., False, False, False],
       [20, 1.88, 495, ..., False, False, False],
       [30, 1.88, 495, ..., False, False, False],
       ...,
       [80, 1.78, 113, ..., False, False, False],
       [90, 1.78, 113, ..., False, False, False],
       [100, 1.78, 113, ..., False, False, False]], dtype=object)

In [22]:
y_train

array([0, 0, 0, ..., 1, 1, 1], dtype=int64)

## 5. 模型训练

使用训练集拟合二分类模型。基线：逻辑回归；进阶：随机森林。类别不平衡时使用 `class_weight='balanced'`。

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 基线：逻辑回归
model_lr = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
model_lr.fit(X_train, y_train)

# 进阶：随机森林
model_rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42)
model_rf.fit(X_train, y_train)

# 可选：XGBoost（若已安装）
try:
    import xgboost as xgb
    model_xgb = xgb.XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1), random_state=42, use_label_encoder=False, eval_metric='logloss')
    model_xgb.fit(X_train, y_train)
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    model_xgb = None

print("训练完成。模型: LogisticRegression, RandomForest" + (", XGBoost" if HAS_XGB else ""))

D:\Programs\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
训练完成。模型: LogisticRegression, RandomForest


In [24]:
model_lr

LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)

In [25]:
model_rf

RandomForestClassifier(class_weight='balanced', max_depth=12, random_state=42)

## 6. 模型评估

在验证集与测试集上计算准确率、精确率、召回率、F1、AUC-ROC，并输出混淆矩阵与分类报告。

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

def evaluate_model(model, X, y, name=""):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    print(f"--- {name} ---")
    print(f"准确率 Accuracy:  {accuracy_score(y, y_pred):.4f}")
    print(f"精确率 Precision: {precision_score(y, y_pred, zero_division=0):.4f}")
    print(f"召回率 Recall:    {recall_score(y, y_pred, zero_division=0):.4f}")
    print(f"F1:              {f1_score(y, y_pred, zero_division=0):.4f}")
    print(f"AUC-ROC:         {roc_auc_score(y, y_prob):.4f}")
    print("混淆矩阵 Confusion Matrix:")
    print(confusion_matrix(y, y_pred))
    print("分类报告 Classification Report:")
    print(classification_report(y, y_pred, target_names=["不受伤", "受伤"], zero_division=0))

print("验证集 评估")
evaluate_model(model_lr, X_val, y_val, "LogisticRegression")
print()
evaluate_model(model_rf, X_val, y_val, "RandomForest")
if HAS_XGB and model_xgb is not None:
    print()
    evaluate_model(model_xgb, X_val, y_val, "XGBoost")

print("\n" + "=" * 50 + "\n测试集 评估")
evaluate_model(model_rf, X_test, y_test, "RandomForest (测试集)")

验证集 评估
--- LogisticRegression ---
准确率 Accuracy:  0.6281
精确率 Precision: 0.4509
召回率 Recall:    0.6305
F1:              0.5258
AUC-ROC:         0.6752
混淆矩阵 Confusion Matrix:
[[10676  6351]
 [ 3057  5216]]
分类报告 Classification Report:
              precision    recall  f1-score   support

         不受伤       0.78      0.63      0.69     17027
          受伤       0.45      0.63      0.53      8273

    accuracy                           0.63     25300
   macro avg       0.61      0.63      0.61     25300
weighted avg       0.67      0.63      0.64     25300


--- RandomForest ---
准确率 Accuracy:  0.6434
精确率 Precision: 0.4637
召回率 Recall:    0.5785
F1:              0.5148
AUC-ROC:         0.6700
混淆矩阵 Confusion Matrix:
[[11492  5535]
 [ 3487  4786]]
分类报告 Classification Report:
              precision    recall  f1-score   support

         不受伤       0.77      0.67      0.72     17027
          受伤       0.46      0.58      0.51      8273

    accuracy                           0.64     25300
   macr

### 核心价值：为什么模型具有实战意义？

#### 1. 突破“随机性”的预测增益
* **量化表现：** AUC 约 **0.68**，准确率约 **0.64**。
* **核心结论：** 表现明显优于随机猜测（0.5）。这证明特征（如伤病史、位置、身体指标等）与“未来受伤”之间存在确凿的统计学关联，模型能够捕捉到有效的预警信号。

#### 2. 支撑业务的“风险分层”
* **从判定到评分：** 抛弃非黑即白的 `0/1` 预测，将 `predict_proba` 作为**“受伤风险分数”**。
* **管理应用：** * **高分球员：** 优先安排轮换、深度体检或降低训练负荷。
    * **低分球员：** 维持常规训练。这种**相对风险排序**在日常竞技管理中比单次预测对错更有价值。

#### 3. 低成本、高可扩展的工程基础
* **全流程闭环：** 已建立从数据清洗、标签构造、特征工程到“多窗口预测”的完整可复现框架。
* **迭代潜力：** 无需昂贵的实时监测设备，仅靠基础数据即可运行。未来可无缝接入**比赛负荷、训练量或生理指标**进行增量优化。

---

### 局限性分析：为何指标难以达到极致？

| 局限维度 | 具体原因 | 业务影响 |
| :--- | :--- | :--- |
| **任务难度** | 仅凭静态信息预测未来 N 天的动态突发事件，本身存在极高的预测上限。 | 指标难有爆发式增长。 |
| **数据噪声** | 标签构造基于“健康段/伤病段”的切换，边界定义的模糊性会限制模型精度。 | 存在约一半的误报率（Precision不高）。 |
| **维度缺失** | 缺乏实时训练强度、比赛频次、睡眠及压力等核心动态特征。 | 无法捕捉由“过度疲劳”引发的突发伤病。 |

---

###   落地指引：如何正确使用这些结果？

#### 1. 定位：作为“风险雷达”而非“最终裁判”
不要将其作为强制停赛的唯一标准，而是作为队医和教练组的**辅助决策工具**。结合人工经验，将其看作一种“风险倾向”提醒。

#### 2. 灵活调优阈值
根据业务需求调整 `predict_proba` 的敏感度：
* **宁错杀不漏掉：** 调低阈值（如 0.35–0.45），扩大监控范围，适合高价值球员。
* **减少误报干扰：** 提高阈值，仅针对极高风险人群进行强力干预。

#### 3. 后续优化方向
* **特征补齐：** 引入“近 30 天累计上场时间”等负荷指标。
* **窗口调优：** 尝试针对不同时间窗口（如 30 天 vs 90 天）进行训练，观察是否存在预测表现更稳健的时间区间。

## 7. 预测接口

对外接口：输入**球员姓名**与**预测窗口天数**，返回「是否受伤」(0/1) 与「受伤概率」。推理时伤病史使用**全历史**统计，特征顺序与训练时 `feature_cols` 一致。

In [30]:
def injury_features_full_history(series):
    """整条 injury_series 的伤病史统计（推理时用全历史）。"""
    if not series or len(series) < 2:
        return {'num_injuries': 0, 'total_injury_days': 0, 'avg_injury_days': 0.0, 'max_injury_days': 0,
                'last_injury_days': 0, 'has_injury_history': 0,
                'recent_healthy_days_mean': 0.0, 'recent_healthy_days_std': 0.0,
                'recent_injury_days_mean': 0.0, 'recent_injury_days_std': 0.0}
    injury_days = [series[i] for i in range(1, len(series), 2)]
    healthy_days = [series[i] for i in range(0, len(series), 2) if series[i] != -1]
    n = len(injury_days)
    total = sum(injury_days)
    last_injury = series[-1] if (len(series) - 1) % 2 == 1 else (series[-2] if len(series) >= 2 else 0)
    recent_h = np.mean(healthy_days[-5:]) if healthy_days else 0.0
    recent_h_std = np.std(healthy_days[-5:]) if len(healthy_days) > 1 else 0.0
    recent_i = np.mean(injury_days[-5:]) if injury_days else 0.0
    recent_i_std = np.std(injury_days[-5:]) if len(injury_days) > 1 else 0.0
    return {'num_injuries': n, 'total_injury_days': total, 'avg_injury_days': total / n if n else 0.0,
            'max_injury_days': max(injury_days) if injury_days else 0, 'last_injury_days': last_injury,
            'has_injury_history': 1 if n else 0,
            'recent_healthy_days_mean': recent_h, 'recent_healthy_days_std': recent_h_std,
            'recent_injury_days_mean': recent_i, 'recent_injury_days_std': recent_i_std}


def predict_injury(player_name, horizon_days, model=None):
    """
    预测接口：指定球员在指定天数内是否会受伤。
    参数:
        player_name: 球员姓名（需在 df_raw 中存在）
        horizon_days: 预测窗口天数（如 10, 30, 60, 90）
        model: 分类模型，默认 model_rf
    返回:
        (will_injure, prob): 是否受伤 0/1，受伤概率 0~1
    """
    if model is None:
        model = model_rf
    row = df_raw[df_raw['name'] == player_name]
    if row.empty:
        raise ValueError(f"未找到球员: {player_name}")
    row = row.iloc[0]
    games, minutes = parse_apparence(row['apparence'])
    avg_min = minutes / games if games else 0
    series = row['injury_series'] if isinstance(row['injury_series'], list) else parse_injury_history(row['injury_history'])
    inj = injury_features_full_history(series)
    nat = str(row['nationality']) if pd.notna(row['nationality']) else 'Unknown'
    try:
        nat_enc = le_nat.transform([nat])[0]
    except ValueError:
        nat_enc = 0
    pos_cols = [c for c in feature_cols if c.startswith('pos_')]
    pos_d = {c: (1 if c == 'pos_' + str(row['position']) else 0) for c in pos_cols}
    feat = {'horizon_days': horizon_days, 'height': float(row['height/m']), 'total_games': games, 'total_minutes': minutes,
            'avg_minutes_per_game': avg_min, **inj, 'nationality_enc': nat_enc, **pos_d}
    X_one = np.array([[feat[c] for c in feature_cols]], dtype=float)
    pred = model.predict(X_one)[0]
    prob = model.predict_proba(X_one)[0, 1]
    return int(pred), float(prob)


# 示例调用
player_name = "Ederson"
horizon_days = 90
will_injure, prob = predict_injury(player_name, horizon_days)
print(f"球员: {player_name}, 预测窗口: {horizon_days} 天")
print(f"是否受伤: {'是' if will_injure else '否'} ({will_injure})")
print(f"受伤概率: {prob:.4f}")

球员: Ederson, 预测窗口: 90 天
是否受伤: 是 (1)
受伤概率: 0.6892


找出所有在 n 天内伤病概率 >= p 的球员

In [35]:
HORIZON = 90
THRESHOLD = 0.8
high_risk = []
for name in df_raw['name'].unique():
    try:
        _, prob = predict_injury(name, HORIZON)
        if prob >= THRESHOLD:
            high_risk.append((name, prob))
    except Exception as e:
        pass  # 跳过无法预测的球员
high_risk.sort(key=lambda x: -x[1])
print(f"2025年6月1日起，{HORIZON} 天内伤病概率 >= {THRESHOLD} 的球员（共 {len(high_risk)} 人）:")
for name, prob in high_risk:
    print(f"  {name}: {prob:.4f}")

2025年6月1日起，90 天内伤病概率 >= 0.8 的球员（共 8 人）:
  Filippo Mané: 0.8645
  Luca Raimund: 0.8616
  Assan Ouédraogo: 0.8600
  Kevin Paredes: 0.8477
  Yvandro Borges Sanches: 0.8461
  Carney Chukwuemeka: 0.8202
  Emmanuel Biumla: 0.8195
  Estanis Pedrola: 0.8020


2026年2月，经过人工查验，这些人有7个人在90内受伤